# Visualize the dnabind encoders

Run every registered encoder on a single sequence pair, print the resulting
arrays, and draw an annotated heatmap gallery so the encodings are easy to
compare side by side. Because the sequences are short, every cell value is
small enough to print on the plot.

This is the interactive companion to `examples/visualize_encodings.py` — handy
for building intuition about what each encoder puts into the input tensor.

## Pick a pair

Both sequences are given 5'→3' and must be the same length. Short pairs keep the
printed matrices readable; the default is `seq1=AGCG`, `seq2=CGAT` (L=4).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dnabind.encoders import get_encoder, list_encoders

seq1 = "AGCG"
seq2 = "CGAT"

s1, s2 = seq1.upper(), seq2.upper()
L = len(s1)
if len(s2) != L:
    raise ValueError(f"seq1 and seq2 must be the same length; got {len(s1)} and {len(s2)}")

names = list_encoders()
print(f"seq1 = {s1}    seq2 = {s2}    (L={L}, seq2 reversed = {s2[::-1]})")
print(f"{len(names)} encoders: {names}")

## Run every encoder

Each encoder's output is asserted to match the shape it declares via
`output_shape`, so this cell doubles as a sanity check that catches a broken
encoder.

In [ ]:
outputs = {}
with np.printoptions(precision=2, suppress=True, linewidth=120):
    for name in names:
        encoder = get_encoder(name)
        arr = encoder.encode(s1, s2, L)
        assert arr.shape == encoder.output_shape(L), (
            f"{name}: encode() shape {arr.shape} != output_shape {encoder.output_shape(L)}"
        )
        outputs[name] = arr
        print(f"\n[{name}]  shape={arr.shape}")
        print(arr[0] if arr.shape[0] == 1 else arr)

## Heatmap gallery

One panel per single-channel encoder; multi-channel encoders (e.g. the
dual-channel one-hot) get one panel per channel.

In [ ]:
def panels(name, arr):
    """Yield (title, 2D matrix) panels for one (C, H, W) encoder output."""
    channels = arr.shape[0]
    if channels == 1:
        yield name, arr[0]
    else:
        for c in range(channels):
            yield f"{name} [ch{c}]", arr[c]


tiles = [(title, mat) for name in names for (title, mat) in panels(name, outputs[name])]
ncol = 3
nrow = -(-len(tiles) // ncol)  # ceil division
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3.4, nrow * 3.2))
axes = np.atleast_1d(axes).reshape(-1)

for ax, (title, mat) in zip(axes, tiles):
    im = ax.imshow(mat, cmap="viridis", aspect="equal")
    ax.set_title(title, fontsize=8)
    mid = (float(mat.max()) + float(mat.min())) / 2.0
    for (i, j), v in np.ndenumerate(mat):
        ax.text(
            j, i, f"{v:.2g}",
            ha="center", va="center", fontsize=6,
            color="white" if v < mid else "black",
        )
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axes[len(tiles):]:
    ax.axis("off")

fig.suptitle(f"dnabind encoders — seq1={s1}, seq2={s2} (L={L})", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()